In [2]:
import pandas as pd

In [3]:
gfw_df = pd.read_csv(
    "../data/processed/merged_clean_data.csv"
)

landmark_df = pd.read_csv(
    "../data/processed/landmark_country_summary.csv"
)

In [4]:
print("GFW Shape:", gfw_df.shape)

print("LANDMARK Shape:", landmark_df.shape)

GFW Shape: (4009, 24)
LANDMARK Shape: (48, 5)


In [5]:
gfw_df[[
    "country",
    "year"
]].head(20)

,country,year
0,Afghanistan,2001
1,Afghanistan,2002
2,Afghanistan,2003
3,Afghanistan,2004
4,Afghanistan,2005
5,Afghanistan,2006
6,Afghanistan,2007
7,Afghanistan,2008
8,Afghanistan,2009
9,Afghanistan,2010


In [6]:
country_gfw_summary = (
    gfw_df.groupby("country")
    .agg({
        "tree_cover_loss_ha": "sum",
        "primary_forest_loss_ha": "sum",
        "carbon_gross_emissions_MgCO2e": "sum",
        "extent_2000_ha": "mean",
        "extent_2010_ha": "mean",
        "gain_2000-2012_ha": "mean"
    })
    .reset_index()
)

In [7]:
country_gfw_summary.head()

,country,tree_cover_loss_ha,primary_forest_loss_ha,carbon_gross_emissions_MgCO2e,extent_2000_ha,extent_2010_ha,gain_2000-2012_ha
0,Afghanistan,1912,0,3.681470e+05,205771.0,71786.0,10738.0
1,Aland,17654,0,4.795715e+06,107739.0,103087.0,2583.0
2,Albania,47320,0,1.732334e+07,648459.0,588824.0,16468.0
3,Algeria,231651,0,4.493548e+07,1223325.0,821437.0,89147.0
4,Angola,4230235,200544,1.497662e+09,55276135.0,53831806.0,1224131.0


In [9]:
country_intelligence_df = pd.merge(
    country_gfw_summary,
    landmark_df,
    on="country",
    how="left"
)

In [10]:
country_intelligence_df.head(20)

,country,tree_cover_loss_ha,primary_forest_loss_ha,carbon_gross_emissions_MgCO2e,extent_2000_ha,extent_2010_ha,gain_2000-2012_ha,territories,recognized,not_recognized,area_km2
0,Afghanistan,1912,0,3.681470e+05,205771.0,71786.0,10738.0,22.0,0.0,22.0,2.044225e+02
1,Aland,17654,0,4.795715e+06,107739.0,103087.0,2583.0,NaN,NaN,NaN,NaN
2,Albania,47320,0,1.732334e+07,648459.0,588824.0,16468.0,NaN,NaN,NaN,NaN
3,Algeria,231651,0,4.493548e+07,1223325.0,821437.0,89147.0,NaN,NaN,NaN,NaN
4,Angola,4230235,200544,1.497662e+09,55276135.0,53831806.0,1224131.0,NaN,NaN,NaN,NaN
5,Argentina,6968747,482661,1.678739e+09,39069969.0,38368480.0,1107407.0,NaN,NaN,NaN,NaN
6,Australia,9218341,68,2.263333e+09,42281439.0,39481413.0,1600599.0,1510.0,1407.0,103.0,5.451420e+06
7,Austria,439219,0,2.289758e+08,4340284.0,4353244.0,36348.0,NaN,NaN,NaN,NaN
8,Azerbaijan,8334,0,2.464528e+06,1268606.0,1082392.0,30269.0,NaN,NaN,NaN,NaN
9,Bangladesh,261746,8842,1.521878e+08,1938958.0,2216745.0,317862.0,NaN,NaN,NaN,NaN


In [11]:
cotry_intelligence_df["forest_extent_change_pct"] = (
    (
        country_intelligence_df["extent_2010_ha"]
        - country_intelligence_df["extent_2000_ha"]
    )
    /
    country_intelligence_df["extent_2000_ha"]un
) * 100

In [12]:
country_intelligence_df["recognition_ratio"] = (
    country_intelligence_df["recognized"]
    /
    country_intelligence_df["territories"]
)

# Intelligence Feature Engineering

Additional socio-environmental intelligence variables were engineered to support future restoration intelligence analysis.

These engineered indicators combine:
- environmental degradation dynamics,
- forest system change,
- and governance-related territorial recognition metrics.

The goal is to move beyond descriptive environmental dashboards toward integrated socio-environmental intelligence systems capable of supporting restoration planning and ecological decision-making.

In [13]:
country_intelligence_df["restoration_pressure_score"] = (
    (
        country_intelligence_df["tree_cover_loss_ha"] / 1000000
    )
    +
    (
        country_intelligence_df["carbon_gross_emissions_MgCO2e"] / 1000000000
    )
    +
    (
        1 - country_intelligence_df["recognition_ratio"].fillna(0)
    )
)

In [14]:
country_intelligence_df[
    [
        "country",
        "restoration_pressure_score",
        "tree_cover_loss_ha",
        "carbon_gross_emissions_MgCO2e",
        "recognition_ratio"
    ]
].sort_values(
    by="restoration_pressure_score",
    ascending=False
).head(15)

,country,restoration_pressure_score,tree_cover_loss_ha,carbon_gross_emissions_MgCO2e,recognition_ratio
18,Brazil,111.476651,73317081,3.815957e+10,1.000000
122,Russia,104.812516,88831076,1.498144e+10,NaN
25,Canada,87.113652,62647469,2.446618e+10,1.000000
158,United States,69.812016,49462860,1.934916e+10,NaN
68,Indonesia,56.169961,31963106,2.320685e+10,NaN
36,Democratic Republic Of The Congo,35.548322,21072346,1.347598e+10,NaN
29,China,19.761335,12766739,5.994596e+09,NaN
87,Malaysia,16.021953,9513550,5.508403e+09,0.000000
15,Bolivia,13.686663,9778669,3.780268e+09,0.872274
6,Australia,11.549886,9218341,2.263333e+09,0.931788


# Preliminary Restoration Intelligence Interpretation

The restoration pressure score represents an early experimental attempt to combine:
- environmental degradation indicators,
- carbon emissions,
- and governance-related recognition metrics

into a unified socio-environmental intelligence indicator.

The results highlight countries containing:
- large forest systems,
- substantial environmental pressures,
- and complex governance dynamics.

This experimental approach does not represent a validated restoration model, but instead demonstrates how environmental and governance datasets may be combined to support future restoration intelligence systems within the Fynos AI framework.

In [15]:
country_intelligence_df.to_csv(
    "../data/processed/country_intelligence_dataset.csv",
    index=False
)

In [16]:
country_intelligence_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   country                        168 non-null    str    
 1   tree_cover_loss_ha             168 non-null    int64  
 2   primary_forest_loss_ha         168 non-null    int64  
 3   carbon_gross_emissions_MgCO2e  168 non-null    float64
 4   extent_2000_ha                 168 non-null    float64
 5   extent_2010_ha                 168 non-null    float64
 6   gain_2000-2012_ha              168 non-null    float64
 7   territories                    40 non-null     float64
 8   recognized                     40 non-null     float64
 9   not_recognized                 40 non-null     float64
 10  area_km2                       40 non-null     float64
 11  forest_extent_change_pct       163 non-null    float64
 12  recognition_ratio              40 non-null     float64
 13  r